In [1]:
!pip install torch torchvision wandb matplotlib seaborn numpy pandas tqdm plotly thop -q
print("All packages installed successfully.")

All packages installed successfully.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from thop import profile
import numpy as np
import pandas as pd
import wandb
from collections import defaultdict
from tqdm import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12,6)

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
class MyCIFAR10(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.data = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
transform = transforms.Compose([
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5, 0.5, 0.5))
])

training_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Dataset loaded successfully.")
print(f"Training dataset size: {len(training_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Image shape: {training_dataset[0][0].shape}")
print(f"Number of classes: {len(training_dataset.classes)}")


Dataset loaded successfully.
Training dataset size: 50000
Test dataset size: 10000
Image shape: torch.Size([3, 32, 32])
Number of classes: 10


In [5]:
training_size = len(training_dataset)
train_size = int(0.7*training_size)
validation_size = training_size - train_size

train_dataset, validation_dataset = random_split(
    training_dataset, [train_size, validation_size],
    generator=torch.Generator().manual_seed(seed))

print(f"Dataset split done.")
print(f"\nTraining dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Total dataset size: {len(train_dataset) + len(validation_dataset) + len(test_dataset)}")

Dataset split done.

Training dataset size: 35000
Validation dataset size: 15000
Test dataset size: 10000
Total dataset size: 60000


In [6]:
train_loader = DataLoader(training_dataset, batch_size=64, shuffle=True,pin_memory=True)
val_loader = DataLoader(validation_dataset, batch_size=64, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False,pin_memory=True)

print(" Dataloaders are ready.")

 Dataloaders are ready.


In [7]:
class NNmodel(nn.Module):
    def __init__(self):
        super(NNmodel, self).__init__()
        # Layer 1: Input 3x32x32 -> Output 32x32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        # Layer 2: Input 32x32x32 -> Output 64x16x16 (after pool)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Layer 3: Input 64x16x16 -> Output 128x8x8 (after pool)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # Fully Connected Layers
        # 128 channels * 4x4 spatial size (after 3 pools: 32->16->8->4)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10) # 10 classes for CIFAR-10
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # Output: 32x16x16
        x = self.pool(F.relu(self.conv2(x))) # Output: 64x8x8
        x = self.pool(F.relu(self.conv3(x))) # Output: 128x4x4

        x = x.view(-1, 128 * 4 * 4) # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = NNmodel()
print(model)

NNmodel(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)


In [8]:
# FLOPs Calculation (THOP)
model.to(device)
dummy_input = torch.randn(1, 3, 32, 32).to(device)

# Now both model and input are on the same device (CUDA)
flops, params = profile(model, inputs=(dummy_input, ))
print(f"FLOPs: {flops}, Params: {params}")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
FLOPs: 10848768.0, Params: 620362.0


In [9]:
wandb.init(
    project="cifar10-lab2",
    name="Jyoti_Dwivedi_M25CSA010_Lab2",
    config={"epochs": 30, "lr": 0.001}
)

wandb.watch(model, log="all", log_freq=10)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    # --- TRAINING PHASE ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Training]"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad(): # Disable gradient calculation for validation
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_loss = criterion(outputs, labels)

            val_loss += v_loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    # Log both to Wandb
    metrics = {
        "epoch": epoch + 1,
        "train_loss": train_loss / len(train_loader),
        "train_acc": 100. * train_correct / train_total,
        "val_loss": val_loss / len(val_loader),
        "val_acc": 100. * val_correct / val_total
    }
    wandb.log(metrics)

    print(f"Epoch {epoch+1}: Train Acc: {metrics['train_acc']:.2f}% | Val Acc: {metrics['val_acc']:.2f}%")

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dwivedijyoti620 (dwivedijyoti620-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 [Training]: 100%|██████████| 782/782 [00:24<00:00, 32.40it/s]


Epoch 1: Train Acc: 43.62% | Val Acc: 55.71%


Epoch 2 [Training]: 100%|██████████| 782/782 [00:22<00:00, 34.27it/s]


Epoch 2: Train Acc: 56.91% | Val Acc: 61.05%


...

Epoch 29 [Training]: 100%|██████████| 782/782 [00:24<00:00, 32.26it/s]


Epoch 29: Train Acc: 84.30% | Val Acc: 87.94%


Epoch 30 [Training]: 100%|██████████| 782/782 [00:23<00:00, 32.79it/s]


Epoch 30: Train Acc: 84.29% | Val Acc: 88.67%


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████████
train_loss,█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████
val_loss,█▇▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,30
train_acc,84.29
train_loss,0.43881
val_acc,88.67333
val_loss,0.32196


In [11]:
# Create a fresh, clean model instance
test_model = NNmodel().to(device)

#  Load the weights
test_model.load_state_dict(model.state_dict(), strict=False)
test_model.eval()

test_correct = 0
test_total = 0

print("Evaluating on the Test Set (10,000 images)...")

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = test_model(images)

        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

final_test_acc = 100 * test_correct / test_total

print(f"\n" + "="*30)
print(f" FINAL TEST ACCURACY: {final_test_acc:.2f}%")
print("="*30)

Evaluating on the Test Set (10,000 images)...


Final Testing: 100%|██████████| 157/157 [00:06<00:00, 24.72it/s]


 FINAL TEST ACCURACY: 75.57%
